# <font color="#418FDE" size="6.5" uppercase>**Cluster und PCA**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Implementieren K-Means-Schritte mit Distanzen, Zuordnung und Zentrenaktualisierung. 
- Berechnen Hauptkomponenten, erklärte Varianz und Projektionen mit NumPy. 
- Analysieren Clusterqualität, Gegenbeispiele und einfache Anomaliehinweise. 


## **1. K Means manuell**

### **1.1. Distanzen berechnen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_01_01.jpg?v=1787641236" width="250">



>* Distanzen messen Ähnlichkeit zu Clusterzentren
>* Kleine Abstände bedeuten ähnliche Datenpunkte

>* Euklidische Distanz misst Nähe im Merkmalsraum
>* Skalierung verhindert dominierende Merkmale

>* Jeder Punkt wird mit allen Zentren verglichen
>* Merkmale bestimmen die Aussagekraft der Distanzen



In [ ]:
#@title Python-Code - Distanzen berechnen

# Wir berechnen Distanzen für K Means.
# Euklidische Abstände bestimmen die nächste Mitte.
# Die Grafik zeigt Punkte und Zentren.

import numpy as np
import matplotlib.pyplot as plt

# Kleine zweidimensionale Datenpunkte bleiben gut nachvollziehbar.
points = np.array([[1.0, 2.0], [2.0, 1.0], [4.0, 4.0], [5.0, 3.0]])
centers = np.array([[1.0, 1.0], [5.0, 4.0]])

# Diese Prüfung verhindert unpassende Punktdimensionen.
if points.shape[1] != centers.shape[1]:
    raise ValueError("Punkte und Zentren brauchen gleich viele Merkmale.")

# Broadcasting bildet alle Punkt-Zentrum-Differenzen gleichzeitig.
differences = points[:, np.newaxis, :] - centers[np.newaxis, :, :]
squared_differences = differences ** 2

# Die euklidische Distanz ist die Wurzel der Quadratsumme.
distances = np.sqrt(np.sum(squared_differences, axis=2))
nearest_center = np.argmin(distances, axis=1)

# Kurze Ausgabe zeigt die entscheidende K-Means-Zwischenrechnung.
print("Distanzen zu Zentrum A und B:")
for index, row in enumerate(np.round(distances, 2)):
    print(f"Punkt {index}: A={row[0]}, B={row[1]}, nächstes={nearest_center[index]}")

# Eine einzelne Grafik verbindet Zahlen mit räumlicher Nähe.
fig, ax = plt.subplots(figsize=(6, 4))
colors = np.array(["tab:blue", "tab:orange"])

ax.scatter(points[:, 0], points[:, 1], c=colors[nearest_center], s=80, label="Punkte")
ax.scatter(centers[:, 0], centers[:, 1], c="black", marker="X", s=160, label="Zentren")

# Dünne Linien zeigen die gemessenen Abstände zum nächsten Zentrum.
for point, center_index in zip(points, nearest_center):
    center = centers[center_index]
    ax.plot([point[0], center[0]], [point[1], center[1]], color="gray", linewidth=1)

ax.set_title("K Means: Distanzen zu aktuellen Zentren")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.show()



### **1.2. Zentren und Zuordnung**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_01_02.jpg?v=1787641238" width="250">



>* Zentren repräsentieren vorläufige Clusterpositionen
>* Punkte wählen das nächstgelegene Zentrum

>* Zentren wandern zum Mittelwert ihrer Punkte
>* K Means bevorzugt kompakte, mittelwertnahe Gruppen

>* Zuordnung und Zentren wandern iterativ zusammen
>* Einfache Nähe kann Cluster verzerren



In [ ]:
#@title Python-Code - Zentren und Zuordnung

# Dieses Beispiel zeigt K-Means-Zuordnung manuell.
# Abstände bestimmen das nächste Zentrum.
# Neue Zentren entstehen aus Cluster-Mittelwerten.

import numpy as np
import matplotlib.pyplot as plt

# Kleine zweidimensionale Punkte bleiben gut überschaubar.
points = np.array([[1.0, 1.0], [1.5, 2.0], [3.0, 4.0], [5.0, 7.0], [6.0, 8.0], [7.0, 8.0]])

# Zwei Startzentren werden bewusst einfach gewählt.
centers = np.array([[1.0, 1.0], [7.0, 8.0]])

# Die Formprüfung verhindert unpassende Punkt-Zentrum-Kombinationen.
if points.shape[1] != centers.shape[1]:
    raise ValueError("Punkte und Zentren brauchen gleich viele Merkmale.")

# Broadcasting berechnet alle Punkt-Zentrum-Abstände gleichzeitig.
differences = points[:, np.newaxis, :] - centers[np.newaxis, :, :]
distances = np.sqrt(np.sum(differences ** 2, axis=2))

# Jeder Punkt erhält das Zentrum mit dem kleinsten Abstand.
assignments = np.argmin(distances, axis=1)

# Für jedes Cluster wird der Mittelwert seiner Punkte berechnet.
new_centers = centers.copy()
for cluster_id in range(len(centers)):
    cluster_points = points[assignments == cluster_id]
    if len(cluster_points) > 0:
        new_centers[cluster_id] = cluster_points.mean(axis=0)

print("Startzentren:", np.round(centers, 2).tolist())
print("Zuordnung je Punkt:", assignments.tolist())
print("Neue Zentren:", np.round(new_centers, 2).tolist())

# Die Grafik zeigt Punkte, Startzentren und aktualisierte Zentren.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(points[:, 0], points[:, 1], c=assignments, s=80, label="Datenpunkte")
ax.scatter(centers[:, 0], centers[:, 1], marker="x", s=140, c="black", label="Startzentren")
ax.scatter(new_centers[:, 0], new_centers[:, 1], marker="*", s=180, c="red", label="Neue Zentren")

ax.set_title("K Means: Zuordnung und Zentrenaktualisierung")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.show()



### **1.3. K Means Iterationen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_01_03.jpg?v=1787641240" width="250">



>* Punkte zuordnen, Zentren neu berechnen
>* Wiederholungen stabilisieren die Cluster schrittweise

>* Änderungen zeigen Stabilität oder Anpassung
>* Cluster brauchen fachliche Interpretation

>* Konvergenz garantiert keine optimale Clusterlösung
>* Startwerte, Datenform und Ausreißer prüfen



In [ ]:
#@title Python-Code - K Means Iterationen

# Dieses Beispiel zeigt K Means Iterationen manuell.
# Distanzen bestimmen Zuordnungen zu aktuellen Zentren.
# Aktualisierte Zentren wandern schrittweise zu Clustern.

import numpy as np
import matplotlib.pyplot as plt

# Kleine zweidimensionale Datenpunkte bleiben gut nachvollziehbar.
points = np.array(
    [[1.0, 1.0], [1.5, 2.0], [3.0, 4.0], [5.0, 7.0], [6.0, 8.0], [7.0, 8.0]]
)

# Zwei Startzentren werden bewusst einfach gewählt.
centers = np.array([[1.0, 1.0], [7.0, 8.0]])

# Diese Prüfung verhindert unpassende Formen für Distanzen.
if points.shape[1] != centers.shape[1]:
    raise ValueError("Punkte und Zentren brauchen gleich viele Merkmale.")

print("Manuelle K Means Iterationen")
print(f"Startzentren: {np.round(centers, 2).tolist()}")

# Drei Iterationen reichen für dieses kleine Beispiel.
for iteration in range(1, 4):
    differences = points[:, np.newaxis, :] - centers[np.newaxis, :, :]
    distances = np.sqrt(np.sum(differences ** 2, axis=2))

    labels = np.argmin(distances, axis=1)
    new_centers = centers.copy()

    for cluster_id in range(len(centers)):
        cluster_points = points[labels == cluster_id]
        if len(cluster_points) > 0:
            new_centers[cluster_id] = cluster_points.mean(axis=0)

    movement = np.sqrt(np.sum((new_centers - centers) ** 2, axis=1))
    print(f"Iteration {iteration}: Labels {labels.tolist()}, Bewegung {np.round(movement, 2).tolist()}")

    centers = new_centers

# Die letzte Zuordnung wird für die Grafik verwendet.
final_labels = labels

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(points[:, 0], points[:, 1], c=final_labels, s=80, cmap="viridis")
ax.scatter(centers[:, 0], centers[:, 1], c="red", marker="X", s=180, label="Zentren")
ax.set_title("K Means nach drei manuellen Iterationen")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.show()



## **2. Cluster bewerten**

### **2.1. Inertia verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_02_01.jpg?v=1787641230" width="250">



>* Inertia misst die Kompaktheit von Clustern
>* Sie bewertet Nähe, nicht fachliche Sinnhaftigkeit

>* Mehr Cluster senken Inertia meist
>* Bewerte auch Sinn, Stabilität und Nutzen

>* Skalierung und Datenform beeinflussen Inertia stark
>* PCA hilft, ersetzt aber keine Prüfung



In [ ]:
#@title Python-Code - Inertia verstehen

# Dieses Beispiel zeigt Inertia bei K-Means.
# Wir vergleichen kompakte und verstreute Cluster.
# Die Grafik macht Abstände zum Zentrum sichtbar.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import sklearn

# Kleine Beispieldaten bleiben übersichtlich und deterministisch.
points = np.array([
    [1.0, 1.0], [1.2, 0.9], [0.8, 1.1], [1.1, 1.2],
    [5.0, 5.0], [5.4, 4.7], [4.6, 5.3], [6.2, 3.8],
])

# Die Formprüfung verhindert stille Fehler bei den Koordinaten.
if points.shape != (8, 2):
    raise ValueError("Die Beispieldaten müssen acht zweidimensionale Punkte enthalten.")

# K-Means findet zwei Zentren und ordnet jeden Punkt zu.
model = KMeans(n_clusters=2, n_init=10, random_state=42)
labels = model.fit_predict(points)
centers = model.cluster_centers_

# Inertia ist die Summe der quadrierten Abstände zum eigenen Zentrum.
assigned_centers = centers[labels]
squared_distances = np.sum((points - assigned_centers) ** 2, axis=1)
manual_inertia = np.sum(squared_distances)

# Kurze Ausgaben verbinden Formel und Modellwert.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Manuell berechnete Inertia: {manual_inertia:.2f}")
print(f"KMeans-Inertia aus dem Modell: {model.inertia_:.2f}")
print(f"Größter einzelner Beitrag: {np.max(squared_distances):.2f}")

# Eine einzelne Grafik zeigt Punkte, Zentren und Abstandsbeiträge.
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(points[:, 0], points[:, 1], c=labels, s=80, cmap="viridis", label="Punkte")
ax.scatter(centers[:, 0], centers[:, 1], c="red", marker="X", s=180, label="Zentren")

# Linien machen sichtbar, welche Abstände Inertia aufsummiert.
for point, center in zip(points, assigned_centers):
    ax.plot([point[0], center[0]], [point[1], center[1]], color="gray", alpha=0.6)

ax.set_title("Inertia: quadrierte Abstände zu Clusterzentren")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.show()



### **2.2. Gegenbeispiele erkennen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_02_02.jpg?v=1787641232" width="250">



>* PCA-Bilder können Cluster überzeugend, aber trügerisch zeigen
>* Gegenbeispiele prüfen fachliche Sinnhaftigkeit von Clustern

>* PCA-Projektionen zeigen nur verdichtete Datensichten
>* Gegenbeispiele prüfen fachlich relevante Unterschiede

>* Gegenbeispiele mit Merkmalen und Kontext prüfen
>* Clusterbewertung braucht Zahlen, Geometrie und Fachwissen



In [ ]:
#@title Python-Code - Gegenbeispiele erkennen

# Dieses Beispiel prüft Gegenbeispiele in einer PCA-Projektion.
# Es vergleicht zweidimensionale Nähe mit ursprünglicher Distanz.
# Sichtbar werden scheinbar nahe, aber unähnliche Punkte.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import sklearn

# Kleine synthetische Daten zeigen eine kontrollierte Cluster-Situation.
rng = np.random.default_rng(42)
cluster_a = rng.normal(loc=[0, 0, 0, 0], scale=0.45, size=(18, 4))
cluster_b = rng.normal(loc=[3, 3, 0, 0], scale=0.45, size=(18, 4))

# Zwei Gegenbeispiele wirken in PC1 und PC2 ähnlich.
tricky_points = np.array([[1.5, 1.5, 3.2, -3.2], [1.6, 1.4, -3.1, 3.1]])
X = np.vstack([cluster_a, cluster_b, tricky_points])

# Standardisierung verhindert Dominanz durch unterschiedliche Skalen.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA wird hier direkt mit NumPy berechnet.
centered = X_scaled - X_scaled.mean(axis=0)
covariance = np.cov(centered, rowvar=False)

# Eigenwerte sortieren die Richtungen nach erklärter Varianz.
eigenvalues, eigenvectors = np.linalg.eigh(covariance)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

# Die ersten zwei Hauptkomponenten bilden die Projektion.
components = eigenvectors[:, :2]
projected = centered @ components
explained_ratio = eigenvalues / eigenvalues.sum()

# K-Means liefert Clusterlabels in der zweidimensionalen Ansicht.
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans.fit_predict(projected)

# Wir suchen das Paar mit kleiner Projektionsdistanz.
projection_distances = np.linalg.norm(projected[:, None, :] - projected[None, :, :], axis=2)
original_distances = np.linalg.norm(X_scaled[:, None, :] - X_scaled[None, :, :], axis=2)

# Diagonale ausschließen, damit kein Punkt sich selbst wählt.
np.fill_diagonal(projection_distances, np.inf)
pair_index = np.unravel_index(np.argmin(projection_distances), projection_distances.shape)
first_index = int(pair_index[0])
second_index = int(pair_index[1])

# Kurze Ausgaben verbinden PCA, Varianz und Gegenbeispiel.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Erklärte Varianz PC1+PC2: {explained_ratio[:2].sum():.2f}")
print(f"Nächstes Paar in der Projektion: {first_index} und {second_index}")
print(f"Distanz in PC1/PC2: {projection_distances[pair_index]:.2f}")
print(f"Distanz im ursprünglichen Raum: {original_distances[pair_index]:.2f}")

# Ein Plot markiert das verdächtige Paar sichtbar.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(projected[:, 0], projected[:, 1], c=labels, cmap="viridis", s=55)

# Die roten Ringe zeigen das erkannte Gegenbeispiel-Paar.
ax.scatter(projected[[first_index, second_index], 0], projected[[first_index, second_index], 1], facecolors="none", edgecolors="red", s=180, linewidths=2, label="prüfen")
ax.set_title("Gegenbeispiele in einer PCA-Projektion erkennen")
ax.set_xlabel("Hauptkomponente 1")
ax.set_ylabel("Hauptkomponente 2")
ax.legend()
plt.show()



### **2.3. Anomalien erkennen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_02_03.jpg?v=1787641234" width="250">



>* PCA zeigt ungewöhnlich entfernte Datenpunkte.
>* Anomalien fachlich prüfen, nicht vorschnell verwerfen.

>* Große Abstände und Kleingruppen zeigen Auffälligkeiten
>* PCA-Projektionen immer mit Originalmerkmalen prüfen

>* Anomalien fachlich und rechnerisch prüfen
>* K-Means-Grenzen als Fragen verstehen



In [ ]:
#@title Python-Code - Anomalien erkennen

# Wir erkennen Anomalien mit PCA und Clusterabständen.
# Hauptkomponenten verdichten die wichtigsten Variationsrichtungen.
# Auffällige Punkte erscheinen weit vom Clusterzentrum.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import sklearn

# Kleine Beispieldaten zeigen zwei Gruppen und zwei Ausreißer.
normal_points = np.array([
    [1.0, 1.2, 0.9], [1.2, 0.8, 1.1], [0.8, 1.1, 1.0],
    [5.0, 5.2, 4.9], [5.3, 4.8, 5.1], [4.7, 5.1, 5.0],
])

anomaly_points = np.array([[8.5, 1.0, 7.5], [0.5, 6.8, 0.2]])
data = np.vstack((normal_points, anomaly_points))
labels_for_plot = np.array(["normal"] * 6 + ["auffällig"] * 2)

# Skalierung verhindert, dass ein Merkmal dominiert.
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data)

# PCA wird hier direkt mit NumPy berechnet.
centered_data = scaled_data - scaled_data.mean(axis=0)
covariance_matrix = np.cov(centered_data, rowvar=False)

# Eigenwerte sortieren die Hauptkomponenten nach Varianz.
eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]

# Die ersten zwei Komponenten bilden die Projektion.
eigenvectors = eigenvectors[:, order]
projected_data = centered_data @ eigenvectors[:, :2]
explained_ratio = eigenvalues / eigenvalues.sum()

# K-Means liefert Clusterzentren in der PCA-Ebene.
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(projected_data)

# Große Abstände zum Zentrum markieren mögliche Anomalien.
centers = kmeans.cluster_centers_
distances = np.linalg.norm(projected_data - centers[cluster_labels], axis=1)
threshold = np.percentile(distances, 75)
anomaly_mask = distances > threshold

# Kurze Ausgaben verbinden PCA-Varianz und Anomaliehinweis.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Erklärte Varianz PC1: {explained_ratio[0]:.2f}")
print(f"Erklärte Varianz PC2: {explained_ratio[1]:.2f}")
print(f"Markierte Punkte: {np.where(anomaly_mask)[0].tolist()}")

# Die Grafik zeigt Projektion, Cluster und auffällige Punkte.
fig, ax = plt.subplots(figsize=(7, 5))
colors = np.where(labels_for_plot == "auffällig", "crimson", "steelblue")
ax.scatter(projected_data[:, 0], projected_data[:, 1], c=colors, s=80)

ax.scatter(centers[:, 0], centers[:, 1], c="black", marker="X", s=140)
for index, point in enumerate(projected_data):
    ax.text(point[0] + 0.05, point[1] + 0.05, str(index))

ax.set_title("Anomalien in einer PCA-Projektion")
ax.set_xlabel("Hauptkomponente 1")
ax.set_ylabel("Hauptkomponente 2")
ax.legend(["Datenpunkte", "Clusterzentren"], loc="best")
plt.show()



## **3. PCA Qualität prüfen**

### **3.1. Kovarianz und Eigenvektoren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_03_01.jpg?v=1787641246" width="250">



>* Kovarianz zeigt gemeinsame Merkmalsbewegungen.
>* Kovarianzmatrix beschreibt wichtige Datenrichtungen.

>* Eigenvektoren zeigen wichtigste Streurichtungen der Daten
>* Plausible Komponenten vermeiden irreführende Muster

>* PCA kann wichtige Cluster verdecken.
>* Ausreißer können Hauptachsen stark verzerren.



In [ ]:
#@title Python-Code - Kovarianz und Eigenvektoren

# Dieses Beispiel zeigt Kovarianz und PCA-Richtungen.
# Eigenvektoren beschreiben die wichtigsten Streuungsrichtungen.
# Ausreißer können die erste Richtung sichtbar kippen.

import numpy as np
import matplotlib.pyplot as plt

# Wir erzeugen eine kleine, schräg gestreckte Punktwolke.
rng = np.random.default_rng(42)
base = rng.normal(0.0, 1.0, size=(60, 2))

stretch = np.array([[2.4, 1.2], [0.4, 0.8]])
clean_data = base @ stretch.T
outlier = np.array([[7.0, -4.5]])

# Der Ausreißer wird nur für den Vergleich ergänzt.
data_with_outlier = np.vstack([clean_data, outlier])
center_clean = clean_data.mean(axis=0)
center_outlier = data_with_outlier.mean(axis=0)

if clean_data.shape[1] != 2:
    raise ValueError("Dieses Beispiel erwartet genau zwei Merkmale.")

# Kovarianz und Eigenvektoren werden direkt mit NumPy berechnet.
cov_clean = np.cov(clean_data, rowvar=False)
values_clean, vectors_clean = np.linalg.eigh(cov_clean)
order_clean = np.argsort(values_clean)[::-1]

values_clean = values_clean[order_clean]
vectors_clean = vectors_clean[:, order_clean]
explained_clean = values_clean / values_clean.sum()

# Jetzt wiederholen wir die Rechnung mit einem einzelnen Ausreißer.
cov_outlier = np.cov(data_with_outlier, rowvar=False)
values_outlier, vectors_outlier = np.linalg.eigh(cov_outlier)
order_outlier = np.argsort(values_outlier)[::-1]

values_outlier = values_outlier[order_outlier]
vectors_outlier = vectors_outlier[:, order_outlier]
explained_outlier = values_outlier / values_outlier.sum()

# Das Vorzeichen eines Eigenvektors ist beliebig.
if vectors_clean[0, 0] < 0:
    vectors_clean[:, 0] = -vectors_clean[:, 0]

if vectors_outlier[0, 0] < 0:
    vectors_outlier[:, 0] = -vectors_outlier[:, 0]

angle_clean = np.degrees(np.arctan2(vectors_clean[1, 0], vectors_clean[0, 0]))
angle_outlier = np.degrees(np.arctan2(vectors_outlier[1, 0], vectors_outlier[0, 0]))

print("Kovarianz ohne Ausreißer:", np.round(cov_clean, 2).tolist())
print("Erklärte Varianz PC1 ohne Ausreißer:", round(explained_clean[0], 3))
print("Winkel PC1 ohne Ausreißer:", round(angle_clean, 1), "Grad")
print("Erklärte Varianz PC1 mit Ausreißer:", round(explained_outlier[0], 3))
print("Winkel PC1 mit Ausreißer:", round(angle_outlier, 1), "Grad")

# Die Grafik zeigt, wie die Hauptachse kippen kann.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(clean_data[:, 0], clean_data[:, 1], alpha=0.7, label="normale Punkte")
ax.scatter(outlier[:, 0], outlier[:, 1], color="red", label="Ausreißer")

scale_clean = np.sqrt(values_clean[0]) * 2.0
scale_outlier = np.sqrt(values_outlier[0]) * 2.0
clean_end = center_clean + vectors_clean[:, 0] * scale_clean

outlier_end = center_outlier + vectors_outlier[:, 0] * scale_outlier
ax.plot([center_clean[0], clean_end[0]], [center_clean[1], clean_end[1]], linewidth=3, label="PC1 ohne Ausreißer")
ax.plot([center_outlier[0], outlier_end[0]], [center_outlier[1], outlier_end[1]], linewidth=3, linestyle="--", label="PC1 mit Ausreißer")

ax.set_title("Kovarianz, Eigenvektor und Ausreißereinfluss")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.show()



### **3.2. Projektion und Varianz**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_03_02.jpg?v=1787641242" width="250">



>* Projektion verdichtet Daten mit Informationsverlust
>* Erklärte Varianz zeigt bewahrte Struktur

>* PCA-Projektionen machen Cluster oft sichtbar
>* Varianz zeigt nicht immer relevante Trennung

>* Varianz immer kritisch im Kontext prüfen
>* PCA zeigt Anomalien, ersetzt keine Bewertung



In [ ]:
#@title Python-Code - Projektion und Varianz

# Wir prüfen PCA-Projektion und erklärte Varianz.
# Ein Gegenbeispiel zeigt versteckte Clustertrennung.
# Die Grafik macht den Informationsverlust sichtbar.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Diese Daten haben starke Streuung und kleine Gruppentrennung.
rng = np.random.default_rng(42)
wide_noise = rng.normal(0.0, 4.0, 80)
small_signal = np.concatenate((rng.normal(-0.7, 0.18, 40), rng.normal(0.7, 0.18, 40)))

# Die Gruppen unterscheiden sich nur in der zweiten Richtung.
labels = np.array([0] * 40 + [1] * 40)
data = np.column_stack((wide_noise, small_signal))

# Eine einfache Prüfung schützt vor falschen Formen.
if data.shape != (80, 2):
    raise ValueError("Die Beispieldaten sollten 80 Zeilen und 2 Spalten haben.")

# Standardisierung verhindert Dominanz durch unterschiedliche Einheiten.
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data)

# PCA sucht Varianzrichtungen, nicht automatisch Clustergrenzen.
pca = PCA(n_components=2, random_state=42)
projected = pca.fit_transform(scaled_data)

# Die erste Komponente erklärt viel, trennt aber schwach.
variance_ratio = pca.explained_variance_ratio_
mean_pc1_gap = abs(projected[labels == 0, 0].mean() - projected[labels == 1, 0].mean())

# Die zweite Komponente erklärt weniger, trennt hier stärker.
mean_pc2_gap = abs(projected[labels == 0, 1].mean() - projected[labels == 1, 1].mean())
print(f"scikit-learn-Version: {sklearn_version}")
print(f"Erklärte Varianz PC1: {variance_ratio[0]:.2f}")
print(f"Erklärte Varianz PC2: {variance_ratio[1]:.2f}")
print(f"Mittlerer Gruppenabstand auf PC1: {mean_pc1_gap:.2f}")
print(f"Mittlerer Gruppenabstand auf PC2: {mean_pc2_gap:.2f}")

# Die Projektion zeigt, welche Achse die Gruppen sichtbar macht.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(projected[:, 0], projected[:, 1], c=labels, cmap="coolwarm", s=45)

# Achsenbeschriftungen verbinden Projektion und Varianzanteile.
ax.set_title("PCA-Projektion: Varianz ist nicht gleich Clustertrennung")
ax.set_xlabel(f"PC1 ({variance_ratio[0]:.0%} erklärte Varianz)")
ax.set_ylabel(f"PC2 ({variance_ratio[1]:.0%} erklärte Varianz)")

# Die Legende benennt die zwei künstlichen Gruppen.
legend = ax.legend(*scatter.legend_elements(), title="Gruppe")
ax.add_artist(legend)
plt.show()



### **3.3. PCA Mini Projekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_B/image_03_03.jpg?v=1787641244" width="250">



>* PCA reduziert Merkmale und zeigt mögliche Cluster.
>* Prüfe Skalierung, Ausreißer und fachliche Plausibilität.

>* Erklärte Varianz vor Interpretation prüfen
>* Cluster fachlich prüfen, Artefakte erkennen

>* Entfernte PCA-Punkte als Anomaliehinweise prüfen
>* Auffälligkeiten begründet statt vorschnell bewerten



In [ ]:
#@title Python-Code - PCA Mini Projekt

# Dieses Mini Projekt prüft PCA Qualität praktisch.
# Wir vergleichen Cluster, Varianz und Anomaliehinweise.
# Die Grafik zeigt Projektion und auffällige Punkte.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Wir erzeugen kleine Daten mit drei plausiblen Gruppen.
features, true_groups = make_blobs(
    n_samples=180, centers=3, n_features=5, cluster_std=1.2, random_state=42
)

# Zwei künstliche Ausreißer simulieren ungewöhnliche Beobachtungen.
outliers = np.array([[8.0, 8.0, 8.0, 8.0, 8.0], [-8.0, -7.0, -8.0, -7.0, -8.0]])
features = np.vstack([features, outliers])

# Eine einfache Prüfung verhindert unerwartete Datenprobleme.
if features.shape != (182, 5) or not np.isfinite(features).all():
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Skalierung verhindert, dass ein Merkmal PCA dominiert.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# PCA verdichtet fünf Merkmale auf zwei sichtbare Achsen.
pca = PCA(n_components=2, random_state=42)
projected = pca.fit_transform(scaled_features)

# K-Means bewertet, ob die Projektion klare Gruppen zeigt.
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(projected)

# Silhouette misst grob die Trennung der gefundenen Cluster.
silhouette = silhouette_score(projected, cluster_labels)
explained = pca.explained_variance_ratio_.sum()

# Große Entfernung vom Zentrum liefert einfache Anomaliehinweise.
center = projected.mean(axis=0)
distances = np.linalg.norm(projected - center, axis=1)
threshold = np.percentile(distances, 98)
anomaly_mask = distances >= threshold

print(f"scikit-learn Version: {sklearn_version}")
print(f"Erklärte Varianz durch PC1 und PC2: {explained:.2%}")
print(f"Silhouette in der PCA-Projektion: {silhouette:.2f}")
print(f"Auffällige Punkte nach Distanzregel: {int(anomaly_mask.sum())}")
print("Merksatz: Hohe Varianz ersetzt keine fachliche Prüfung.")

# Die Grafik verbindet Clusterbild und Anomaliehinweise.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    projected[:, 0], projected[:, 1], c=cluster_labels, cmap="viridis", alpha=0.75
)

# Auffällige Punkte werden zusätzlich rot umrandet.
ax.scatter(
    projected[anomaly_mask, 0], projected[anomaly_mask, 1],
    facecolors="none", edgecolors="red", s=140, linewidths=2, label="auffällig"
)

ax.set_title("PCA Mini Projekt: Clusterqualität und Anomalien")
ax.set_xlabel("Hauptkomponente 1")
ax.set_ylabel("Hauptkomponente 2")
ax.legend(loc="best")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Cluster und PCA**</font>


In this lecture, you learned to:
- Implementieren K-Means-Schritte mit Distanzen, Zuordnung und Zentrenaktualisierung. 
- Berechnen Hauptkomponenten, erklärte Varianz und Projektionen mit NumPy. 
- Analysieren Clusterqualität, Gegenbeispiele und einfache Anomaliehinweise. 

In the next Module (Module 9), we will go over 'scikit-learn Start'